# HamEvo Inference Demo

This notebook loads one HamEvo checkpoint and runs one of two inference modes:

1. Provide an `xyz` file path and predict Hamiltonians/orbital properties.
2. Provide a `gdb17-scf-rerun` root path plus an integer index and compare against dataset ground truth.

## User Inputs

- `CKPT_PATH`: checkpoint file path. Absolute paths and repo-relative paths both work.
- `XYZ_PATH`: xyz file path for xyz inference. Coordinates are interpreted as Angstrom, matching the training/evaluation data pipeline.
- `GDB17_ROOT` and `GDB17_IDX`: dataset root and sample index for GDB17 inference.

In [ ]:
# User configuration
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from the HamEvo repo, or set REPO_ROOT to the repo root.")

# User inputs. Absolute paths and repo-relative paths both work.
# XYZ coordinates are interpreted as Angstrom.
CKPT_PATH = Path("path/to/checkpoint.ckpt")
XYZ_PATH = Path("path/to/input.xyz")
GDB17_ROOT = Path("path/to/gdb17-scf-rerun")
GDB17_IDX = 399

DEVICE = "cuda"
DATA_TYPE = "float32"
FULL_ORBITALS = 18
F_THRES = 40
EPS = 1e-6
USE_EMA = True

In [ ]:
# Bootstrap imports
import os
import sys
import time
from contextlib import nullcontext
from typing import Any, Dict, List, Optional, Union

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pyscf
import torch
from torch_geometric.data import Batch
from torch_geometric.loader import DataLoader

from src.data.components.inference import CustomInferData
from src.data.components.lmdb_datasets import GDB17LoopV2
from src.data.components.subset import CustomSubset
from src.models.deqham_module import DEQHamLitModule
from src.utils.physical_cal import (
    cal_orbital_and_energies,
    get_dipole,
    get_mo_occ,
    get_properties_error,
    make_rdm1,
    matrix_transform,
)

HARTREE_TO_EV = 27.211386

In [ ]:
# Common helpers. These should move to src/utils/inference_api.py.

def set_inference_precision(tf32: bool = False) -> None:
    torch.backends.cuda.matmul.allow_tf32 = bool(tf32)
    torch.backends.cudnn.allow_tf32 = bool(tf32)
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high" if tf32 else "highest")


def resolve_user_path(path: Union[str, Path]) -> Path:
    path = Path(path).expanduser()
    return path if path.is_absolute() else REPO_ROOT / path


def require_existing_path(path: Union[str, Path], label: str) -> Path:
    path = resolve_user_path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label} not found: {path}")
    return path


def load_hamevo_model(ckpt_path: Union[str, Path], device: str = "cuda"):
    ckpt_path = require_existing_path(ckpt_path, "Checkpoint")
    model = DEQHamLitModule.load_from_checkpoint(str(ckpt_path), map_location="cpu", load_ckpt=False)
    model.eval().to(device)
    if hasattr(model, "ema"):
        model.ema.to(device)
    return model


def ema_context(model, use_ema: bool):
    if use_ema and hasattr(model, "ema") and callable(model.ema.average_parameters):
        return model.ema.average_parameters()
    return nullcontext()


def concat_mask(batch):
    return torch.cat([batch.diag_ham_mask, batch.non_diag_ham_mask], dim=0)


def masked_abs_mean(x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    return (x.abs() * mask).sum() / mask.sum()


def solve_fixed_point(
    model,
    batch,
    mode: str,
    f_thres: int = 40,
    eps: float = 1e-6,
    use_ema: bool = True,
    warm_start: Optional[torch.Tensor] = None,
) -> Dict[str, Any]:
    batch = batch.to(model.device)
    batch = model.preprocess(batch, mode=mode)
    if warm_start is not None:
        n_diag = len(batch.diag_ham_mask)
        batch.z["diag_ham"] = warm_start[:n_diag]
        batch.z["non_diag_ham"] = warm_start[n_diag:]

    start_time = time.time()
    with torch.no_grad(), ema_context(model, use_ema):
        result = model.f_solver(
            lambda z: model.func(z, batch),
            model.concat_z(batch["z"]),
            threshold=f_thres,
            eps=eps,
        )
    z_pred = result["result"].detach()
    mask = concat_mask(batch)
    return {
        "batch": batch,
        "z_pred": z_pred,
        "solver": result,
        "nstep": int(result["nstep"]),
        "gx": float(masked_abs_mean(result["gx"], mask).detach().cpu()),
        "elapsed_s": time.time() - start_time,
    }


def build_pred_hamiltonians(model, batch, z_pred):
    n_diag = len(batch.diag_ham_mask)
    hams = model.func.build_final_matrix(batch, z_pred[:n_diag], z_pred[n_diag:])
    return hams if isinstance(hams, list) else list(hams)


def overlap_for_molecule(batch, mol_idx: int):
    if isinstance(batch.ovlp, torch.Tensor):
        ovlp = batch.ovlp.reshape(-1, batch.ovlp.shape[-1], batch.ovlp.shape[-1])[mol_idx]
    else:
        ovlp = torch.tensor(batch.ovlp[mol_idx])
    return ovlp.to(batch.pos.device, batch.pos.dtype)

## Load Model

If this cell fails with an e3nn pickle/codegen assertion, the checkpoint was saved with an incompatible e3nn runtime. Use a checkpoint saved in the release environment or implement a tensor-only fallback loader in `load_hamevo_model`.

In [ ]:
set_inference_precision(tf32=False)
model = load_hamevo_model(CKPT_PATH, device=DEVICE)
print(f"Loaded model from {CKPT_PATH}")
print(f"Device: {model.device}")

In [ ]:
# XYZ input mode.

def custom_infer_args_from_xyz_path(xyz_path: Union[str, Path]) -> Dict[str, Any]:
    xyz_path = require_existing_path(xyz_path, "XYZ file")
    return {
        "root": xyz_path.parent.parent,
        "name": xyz_path.parent.name,
        "xyz_file": xyz_path.name,
    }


def summarize_xyz_molecule(batch, ham_pred, mol_idx: int) -> Dict[str, Any]:
    start, end = batch.ptr[mol_idx].item(), batch.ptr[mol_idx + 1].item()
    atoms = batch.atoms[start:end]
    pos = batch.pos[start:end]
    atoms_np = atoms.squeeze().detach().cpu().numpy()
    pos_np = pos.squeeze().detach().cpu().numpy()

    ham_pyscf = matrix_transform(ham_pred, atoms_np, convention="back2pyscf", tc=True)
    mol = pyscf.gto.Mole()
    mol.build(verbose=0, atom=[[atoms_np[i], pos_np[i]] for i in range(len(atoms_np))], basis="def2svp", unit="ang")

    overlap = overlap_for_molecule(batch, mol_idx)
    orb_ener, orb_coeff = cal_orbital_and_energies(overlap.unsqueeze(0), ham_pyscf.unsqueeze(0))
    mo_occ = get_mo_occ(orb_ener[0], atoms=atoms)
    dm = make_rdm1(orb_coeff[0], mo_occ)
    dipole = get_dipole(dm, mol)
    nocc = int(atoms.sum().item() / 2)
    return {
        "ham_pred": ham_pyscf.detach().cpu(),
        "homo_ev": float(orb_ener[:, nocc - 1].detach().cpu() * HARTREE_TO_EV),
        "lumo_ev": float(orb_ener[:, nocc].detach().cpu() * HARTREE_TO_EV),
        "dipole_debye": float((dipole.norm(dim=-1) * 2.541746).detach().cpu()),
    }


def predict_from_xyz(
    model,
    xyz_path: Union[str, Path],
    batch_size: int = 1,
    f_thres: int = 40,
    eps: float = 1e-6,
    use_ema: bool = True,
    warm_start: bool = True,
) -> List[Dict[str, Any]]:
    infer_args = custom_infer_args_from_xyz_path(xyz_path)
    dataset = CustomInferData(
        root=infer_args["root"],
        name=infer_args["name"],
        xyz_file=infer_args["xyz_file"],
        full_orbitals=FULL_ORBITALS,
        data_type=DATA_TYPE,
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    all_results = []
    last_z_pred = None
    for batch_idx, batch in enumerate(loader):
        solved = solve_fixed_point(
            model,
            batch,
            mode="infer",
            f_thres=f_thres,
            eps=eps,
            use_ema=use_ema,
            warm_start=last_z_pred if warm_start else None,
        )
        batch = solved["batch"]
        z_pred = solved["z_pred"]
        last_z_pred = z_pred.clone()
        hams = build_pred_hamiltonians(model, batch, z_pred)
        for mol_idx, ham_pred in enumerate(hams):
            item = summarize_xyz_molecule(batch, ham_pred, mol_idx)
            item.update({"batch_idx": batch_idx, "mol_idx": mol_idx, "nstep": solved["nstep"], "gx": solved["gx"]})
            all_results.append(item)
    return all_results

In [ ]:
# Run xyz inference after setting XYZ_PATH.
xyz_summary = None
xyz_path = None if XYZ_PATH is None else resolve_user_path(XYZ_PATH)
if xyz_path is None or not xyz_path.exists():
    print(f"Set XYZ_PATH to a real xyz file to run xyz inference: {XYZ_PATH}")
else:
    xyz_results = predict_from_xyz(model, xyz_path, f_thres=F_THRES, eps=EPS, use_ema=USE_EMA)
    xyz_summary = xyz_results[0]
xyz_summary

In [ ]:
# GDB17 index mode.

def predict_from_gdb17_idx(
    model,
    data_root: Union[str, Path],
    idx: int,
    f_thres: int = 40,
    eps: float = 1e-6,
    use_ema: bool = True,
    package: str = "pyscf",
) -> Dict[str, Any]:
    data_root = require_existing_path(data_root, "GDB17 root")
    dataset = GDB17LoopV2(root=data_root, data_type=DATA_TYPE, version="ood", get_ovlp=True)
    subset = CustomSubset(dataset, [idx], mode="predict", dtype=getattr(torch, DATA_TYPE), more_info=False, package=package)
    batch = Batch.from_data_list([subset[0]])
    solved = solve_fixed_point(model, batch, mode="predict", f_thres=f_thres, eps=eps, use_ema=use_ema)
    batch = solved["batch"]
    z_pred = solved["z_pred"]
    z_gt = model.concat_z(batch["z_star"])
    props = get_properties_error(model.func, batch, z_pred, z_gt, batch.get("ovlp", None), package=package)[0]

    coeff_cos = torch.cosine_similarity(props["coeff_pred"], props["coeff_gt"], dim=1).abs().mean()
    ham_mae = (props["ham_pred"] - props["ham_gt"]).abs().mean()
    orb_ener_mae = (props["ener_pred"] - props["ener_gt"]).abs().mean()
    homo_err = (props["HOMO_pred"] - props["HOMO_gt"]).abs().mean()
    lumo_err = (props["LUMO_pred"] - props["LUMO_gt"]).abs().mean()

    return {
        "idx": idx,
        "mol_id": int(batch.mol_id[0]) if torch.is_tensor(batch.mol_id) else int(batch.mol_id),
        "natoms": int(batch.ptr[1] - batch.ptr[0]),
        "ham_pred": props["ham_pred"].detach().cpu(),
        "ham_gt": props["ham_gt"].detach().cpu(),
        "ham_mae": float(ham_mae.detach().cpu()),
        "homo_pred_ev": float((props["HOMO_pred"] * HARTREE_TO_EV).detach().cpu()),
        "homo_gt_ev": float((props["HOMO_gt"] * HARTREE_TO_EV).detach().cpu()),
        "homo_err_ev": float((homo_err * HARTREE_TO_EV).detach().cpu()),
        "lumo_pred_ev": float((props["LUMO_pred"] * HARTREE_TO_EV).detach().cpu()),
        "lumo_gt_ev": float((props["LUMO_gt"] * HARTREE_TO_EV).detach().cpu()),
        "lumo_err_ev": float((lumo_err * HARTREE_TO_EV).detach().cpu()),
        "orbital_energy_mae_ev": float((orb_ener_mae * HARTREE_TO_EV).detach().cpu()),
        "coeff_cosine_similarity": float(coeff_cos.detach().cpu()),
        "dipole_pred": props["dipole_pred"].detach().cpu(),
        "dipole_gt": props["dipole_gt"].detach().cpu(),
        "nstep": solved["nstep"],
        "gx": solved["gx"],
    }

In [ ]:
# Run GDB17 index inference after configuring GDB17_ROOT and GDB17_IDX.
gdb17_summary = None
if not resolve_user_path(GDB17_ROOT).exists():
    print(f"Set GDB17_ROOT to a real dataset directory to run GDB17 inference: {GDB17_ROOT}")
else:
    gdb17_result = predict_from_gdb17_idx(model, GDB17_ROOT, GDB17_IDX, f_thres=F_THRES, eps=EPS, use_ema=USE_EMA)
    gdb17_summary = {k: v for k, v in gdb17_result.items() if k not in {"ham_pred", "ham_gt"}}
gdb17_summary